# 03 — Vektorisierung: Von Text zu Zahlen

**Ziel dieses Notebooks:** Den vorverarbeiteten Text (`train_preprocessed.csv`)
in numerische Feature-Vektoren ueberfuehren, die ein klassisches ML-Modell
(Phase 4) verarbeiten kann. Methoden, Parameter und Textvariante werden
datengetrieben verglichen, nicht angenommen.

**Leitfragen:**
- BoW vs. TF-IDF: welcher Unterschied zeigt sich konkret?
- Wie beeinflussen min_df/max_df die Vokabulargroesse?
- Helfen n-Gramme (Bigramme/Trigramme) ueber Unigramme hinaus?
- Welche der vier Preprocessing-Varianten (clean/no_stopwords/stemmed/
  lemmatized) eignet sich am besten als Vektorisierungs-Basis?
- Wie aehnlich/unaehnlich sind Tweets im Vektorraum (Cosine Similarity)?
- Reduziert LSA/SVD sinnvoll, ohne viel Varianz zu verlieren?
- Wie gut trennen sich die Klassen visuell im reduzierten Raum
  (PCA/t-SNE, 2D) und quantifiziert (Silhouette-Score)?

**Wichtige Einschraenkung:** Die finale Entscheidung "welche Variante/
Parameter performt am besten" wird hier NICHT abschliessend getroffen —
nur Indikatoren erhoben (Dimensionen, Sparsity, Separierbarkeit). Der
belastbare Test mit echtem Modelltraining folgt in Phase 4.

**Output:** finaler Vectorizer + Matrix (fuer Phase 4), Plots/Tabellen
wie gewohnt.

In [ ]:
# =============================================================================
# Zelle 02 – Imports, Store44-Style aktivieren, vorverarbeiteten Datensatz laden
# =============================================================================
# keep_default_na=False: keine automatische NaN-Erkennung beim Einlesen.
# Leere Strings in Text-Spalten sind gueltige Werte (Tweet bestand nur aus
# Stopwords/Mentions). keyword/location werden danach gezielt zurueck zu NaN
# konvertiert, da dort Fehlen inhaltlich bedeutungsvoll ist (Notebook 01).

import sys
sys.path.append("../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from viz_config import (
    apply_store44_style, save_figure,
    COLOR_GOLD, COLOR_BLUE, COLOR_GREEN,
    COLOR_CLASS_0, COLOR_CLASS_1, COLOR_TEXT_MUTED
)

apply_store44_style()

train = pd.read_csv(
    "../data/processed/train_preprocessed.csv",
    keep_default_na=False,
)

train[["keyword", "location"]] = train[["keyword", "location"]].replace("", np.nan)

text_columns = ["text", "text_clean", "text_no_stopwords", "text_stemmed", "text_lemmatized"]
print("Shape:", train.shape)
print("\nNaN-Check Text-Spalten (sollte 0 sein):")
print(train[text_columns].isnull().sum())
print("\nNaN-Check keyword/location (echte Fehlquote, sollte wieder Notebook-01-Werten entsprechen):")
print(train[["keyword", "location"]].isnull().sum())

train[text_columns].head(3)

In [ ]:
# =============================================================================
# Zelle 03 – Bag of Words (CountVectorizer) als Ausgangspunkt
# =============================================================================
# Wir starten mit text_no_stopwords als Arbeitsbasis (neutraler Mittelweg
# zwischen roh und stark transformiert). Vergleich aller 4 Varianten folgt
# spaeter in Zelle 11.

from sklearn.feature_extraction.text import CountVectorizer

bow_vectorizer = CountVectorizer()
X_bow = bow_vectorizer.fit_transform(train["text_no_stopwords"])

print("Shape der BoW-Matrix:", X_bow.shape)
print("Vokabulargroesse:", len(bow_vectorizer.vocabulary_))

# Sparsity: wie viel Prozent der Matrix sind Nullen?
n_total = X_bow.shape[0] * X_bow.shape[1]
n_nonzero = X_bow.nnz
sparsity = (1 - n_nonzero / n_total) * 100
print(f"\nNicht-Null-Eintraege: {n_nonzero:,} von {n_total:,} ({100-sparsity:.3f}%)")
print(f"Sparsity (Anteil Nullen): {sparsity:.3f}%")

# Speicherbedarf: dicht vs. sparse (zeigt, warum sparse-Format essenziell ist)
dense_memory_mb = n_total * 8 / (1024**2)  # 8 Bytes pro float64
sparse_memory_mb = X_bow.data.nbytes / (1024**2)
print(f"\nSpeicherbedarf als DICHTE Matrix: {dense_memory_mb:.1f} MB")
print(f"Speicherbedarf als SPARSE Matrix: {sparse_memory_mb:.1f} MB")
print(f"Faktor: {dense_memory_mb/sparse_memory_mb:.0f}x weniger Speicher durch Sparse-Format")

## Erkenntnis: Bag of Words (Ausgangspunkt)

- Matrix-Shape: 7.434 x 15.803 (Dokumente x Vokabular)
- Sparsity: 99,944% — nur 0,056% der Matrix sind Nicht-Null-Eintraege
- Speicherersparnis durch Sparse-Format: Faktor 1.787x (896,3 MB dicht
  vs. 0,5 MB sparse fuer identischen Inhalt)

**Einordnung:** Die extreme Sparsity ist strukturell erwartbar — jeder
einzelne Tweet enthaelt nur ~9 Woerter (siehe Notebook 02, nach Stopword-
Entfernung), das Gesamtvokabular aber 15.803 Woerter. Jede Zeile der
Matrix hat daher fast ausschliesslich Nullen. Das ist der zentrale Grund,
warum Textverarbeitung mit sparse-Datenstrukturen arbeitet statt mit
regulaeren Arrays — Letzteres waere bei diesem Datensatz bereits jetzt
unhandlich, bei groesseren Korpora schlicht nicht mehr durchfuehrbar.

**Limitation von BoW:** Zaehlt nur Worthaeufigkeiten, gewichtet jedes
Wort gleich — ein 50x vorkommendes, thematisch neutrales Wort (z. B.
"like") hat denselben Einfluss wie ein seltenes, hoch spezifisches Wort
(z. B. "derailment"). TF-IDF (naechste Zelle) adressiert genau das.

In [ ]:
# =============================================================================
# Zelle 05 – TF-IDF (Term Frequency-Inverse Document Frequency)
# =============================================================================

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(train["text_no_stopwords"])

print("Shape der TF-IDF-Matrix:", X_tfidf.shape)
print("Vokabulargroesse:", len(tfidf_vectorizer.vocabulary_))

# Gleiche Dimensionen wie BoW erwartet (gleicher Tokenizer/Vokabular-Aufbau,
# nur die GEWICHTUNG der Eintraege unterscheidet sich)
print("\nShape identisch zu BoW?", X_tfidf.shape == X_bow.shape)

# Konkreter Vergleich an einem Beispieldokument: Rohzaehlung vs. TF-IDF-Gewicht
doc_idx = 2
feature_names = np.array(tfidf_vectorizer.get_feature_names_out())

bow_row = X_bow[doc_idx].toarray().flatten()
tfidf_row = X_tfidf[doc_idx].toarray().flatten()

nonzero_idx = bow_row.nonzero()[0]
comparison_df = pd.DataFrame({
    "word": feature_names[nonzero_idx],
    "bow_count": bow_row[nonzero_idx],
    "tfidf_weight": tfidf_row[nonzero_idx].round(3),
}).sort_values("tfidf_weight", ascending=False)

print(f"\nText: {train['text_no_stopwords'].iloc[doc_idx]}")
print("\nBoW-Zaehlung vs. TF-IDF-Gewicht (sortiert nach TF-IDF):")
print(comparison_df.to_string(index=False))

## Erkenntnis: TF-IDF vs. BoW

Gleiche Matrix-Dimensionen (7.434 x 15.803) — der Unterschied liegt
ausschliesslich in der Gewichtung, nicht im Vokabular.

**Beispiel-Tweet** ("residents asked shelter place notified officers
evacuation shelter place orders expected"):
- "shelter" (2x) erhaelt das hoechste TF-IDF-Gewicht (0,555) — sowohl
  haeufig im Tweet als auch vermutlich seltener im Gesamtkorpus
- "evacuation" hat trotz inhaltlicher Relevanz nur 0,211 — kommt im
  Gesamtkorpus offenbar haeufiger vor (in vielen Disaster-Tweets praesent),
  daher niedrigeres IDF-Gewicht trotz thematischer Bedeutung
- Selbst bei gleichem BoW-Count (1) unterscheiden sich die TF-IDF-Gewichte
  deutlich (0,315 vs. 0,211) — reiner IDF-Effekt (Seltenheit im Korpus)

**Konsequenz:** TF-IDF gewichtet automatisch herunter, was im Korpus
verbreitet ist (auch wenn thematisch relevant), und hebt hervor, was
selten/spezifisch ist. Das ist meist vorteilhaft fuer Klassifikation,
kann aber theoretisch auch thematisch wichtige, aber haeufige Begriffe
unterbewerten — wird in Phase 4 empirisch geprueft (TF-IDF vs. BoW als
Modell-Input im direkten Vergleich).

**Entscheidung:** TF-IDF wird als primaere Vektorisierungsmethode
fuer Phase 4 verwendet (Standardwahl, theoretisch begruendet); BoW
bleibt als Vergleichsoption fuer die Ablationsstudie verfuegbar.

In [ ]:
# =============================================================================
# Zelle 07 – min_df/max_df: Effekt auf Vokabulargroesse
# =============================================================================
# min_df: Wort muss in mindestens N Dokumenten vorkommen (filtert Raritaeten)
# max_df: Wort darf in hoechstens X% der Dokumente vorkommen (filtert
# zu haeufige, wenig spezifische Woerter)

min_df_values = [1, 2, 3, 5, 10, 20]
max_df_values = [1.0, 0.9, 0.7, 0.5, 0.3]

min_df_results = []
for val in min_df_values:
    vec = TfidfVectorizer(min_df=val)
    X = vec.fit_transform(train["text_no_stopwords"])
    min_df_results.append({"min_df": val, "vocab_size": X.shape[1]})

max_df_results = []
for val in max_df_values:
    vec = TfidfVectorizer(max_df=val)
    X = vec.fit_transform(train["text_no_stopwords"])
    max_df_results.append({"max_df": val, "vocab_size": X.shape[1]})

min_df_df = pd.DataFrame(min_df_results)
max_df_df = pd.DataFrame(max_df_results)

print("=== min_df-Effekt ===")
print(min_df_df)
print("\n=== max_df-Effekt ===")
print(max_df_df)

min_df_df.to_csv("../reports/tables/03_min_df_effect.csv", index=False)
max_df_df.to_csv("../reports/tables/03_max_df_effect.csv", index=False)

# Plot: beide Effekte nebeneinander
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(min_df_df["min_df"], min_df_df["vocab_size"], marker="o", color=COLOR_GOLD)
axes[0].set_xlabel("min_df (Mindestanzahl Dokumente)")
axes[0].set_ylabel("Vokabulargroesse")
axes[0].set_title("Effekt von min_df")

axes[1].plot(max_df_df["max_df"], max_df_df["vocab_size"], marker="o", color=COLOR_BLUE)
axes[1].set_xlabel("max_df (max. Dokumentanteil)")
axes[1].set_ylabel("Vokabulargroesse")
axes[1].set_title("Effekt von max_df")
axes[1].invert_xaxis()

fig.suptitle("Vokabulargroesse in Abhaengigkeit von min_df/max_df")
fig.tight_layout()

save_figure(fig, "../reports/figures/03_min_max_df_effect.png")
plt.show()

## Erkenntnis: min_df/max_df-Effekt

**min_df: starker, schnell abflachender Effekt**
- min_df=1 (Standard): 15.803 Woerter
- min_df=2: 6.074 Woerter (-61,6%) — bereits die Haelfte der Reduktion
  durch einen einzigen Schritt erreicht
- min_df=20: nur noch 697 Woerter

Bestaetigt den Heaps'-Law-Befund aus Notebook 01: ein Grossteil des
Vokabulars besteht aus seltenen Einzelvorkommen (84,3% der location-Werte
waren Einzelfaelle, hier zeigt sich ein analoges Muster im Text-Vokabular).

**max_df: praktisch wirkungslos bis 0,3**
Vokabulargroesse bleibt bei 15.803 konstant ueber den gesamten getesteten
Bereich. Erklaerung: Tweets sind sehr kurz (durchschnittlich 9,2 Tokens
nach Stopword-Entfernung, siehe Notebook 02) — kein Wort kommt in mehr
als 30-50% aller 7.434 Dokumente vor. max_df waere erst bei deutlich
laengeren Dokumenten oder einem kleineren, thematisch engeren Korpus
relevant.

**Konsequenz fuer Phase 4:** min_df ist der Hyperparameter mit
tatsaechlichem Einfluss und gehoert ins Grid-Search-Tuning (Phase 5).
max_df kann auf Standardwert (1.0, kein Filter) belassen werden — der
Suchraum sollte sich nicht mit einem wirkungslosen Parameter aufblaehen.

In [ ]:
# =============================================================================
# Zelle 09 – n-Gramme: Unigramm vs. Bigramm vs. Trigramm
# =============================================================================
# n-Gramme erfassen Wortfolgen statt einzelner Woerter, z. B. Bigramm
# "forest fire" als ein Feature statt nur "forest" und "fire" getrennt.

ngram_configs = [
    ("Unigramm (1,1)", (1, 1)),
    ("Bigramm (2,2)", (2, 2)),
    ("Trigramm (3,3)", (3, 3)),
    ("Uni+Bigramm (1,2)", (1, 2)),
    ("Uni+Bi+Trigramm (1,3)", (1, 3)),
]

ngram_results = []
for name, ngram_range in ngram_configs:
    vec = TfidfVectorizer(ngram_range=ngram_range)
    X = vec.fit_transform(train["text_no_stopwords"])
    ngram_results.append({"config": name, "vocab_size": X.shape[1]})

ngram_df = pd.DataFrame(ngram_results)
print(ngram_df)
ngram_df.to_csv("../reports/tables/03_ngram_effect.csv", index=False)

# Beispiel: welche Bigramme sind am haeufigsten?
vec_bigram = TfidfVectorizer(ngram_range=(2, 2), min_df=5)
X_bigram = vec_bigram.fit_transform(train["text_no_stopwords"])
bigram_counts = np.asarray(X_bigram.sum(axis=0)).flatten()
bigram_names = np.array(vec_bigram.get_feature_names_out())
top_bigrams = pd.DataFrame({"bigram": bigram_names, "tfidf_sum": bigram_counts}) \
    .sort_values("tfidf_sum", ascending=False).head(15)
print("\nTop 15 Bigramme (nach TF-IDF-Summe, min_df=5):")
print(top_bigrams.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(ngram_df["config"], ngram_df["vocab_size"], color=COLOR_GOLD)
ax.set_ylabel("Vokabulargroesse")
ax.set_title("Vokabulargroesse je n-Gramm-Konfiguration")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()

save_figure(fig, "../reports/figures/03_ngram_effect.png")
plt.show()

## Erkenntnis: n-Gramme

**Vokabular waechst stark mit hoeherer n-Gramm-Ordnung:**
| Konfiguration | Vokabulargroesse | Faktor ggue. Unigramm |
|---|---|---|
| Unigramm | 15.803 | 1,0x |
| Bigramm | 45.660 | 2,9x |
| Trigramm | 43.824 | 2,8x |
| Uni+Bigramm | 61.463 | 3,9x |
| Uni+Bi+Trigramm | 105.287 | 6,7x |

Klassische Kombinationsexplosion: Bigramme/Trigramme erzeugen deutlich
mehr eindeutige Kombinationen als Unigramme, bei nur 7.434 kurzen Tweets
(~9 Tokens/Tweet) bedeutet das viele Bigramme mit sehr wenigen
Vorkommen (hohe Sparsity, vermutlich aehnlich dem min_df-Befund aus
Zelle 08).

**Top-Bigramme zeigen gemischtes Bild:**
- Inhaltlich aussagekraeftig: "burning buildings", "natural disaster",
  "mass murderer", "suicide bombing", "wild fires", "dust storm",
  "first responders" — echte, thematisch praezise Wortpaare
- Rauschen/Artefakt: **"num num"** an erster Stelle (TF-IDF-Summe 177,
  mit Abstand hoechster Wert) — entsteht durch den NUM-Platzhalter aus
  Notebook 02 (z. B. bei Datums-/Zahlenfolgen wie "8/6/2015" oder
  Telefonnummern, die mehrere aufeinanderfolgende Zahlenblöcke enthalten).
  Traegt keine inhaltliche Bedeutung — analog zum frueheren URL-Befund.
- Themenfremd: "youtube video", "liked youtube" — automatisierte
  Social-Media-Tweets ohne Disaster-Bezug

**Konsequenz:** n-Gramme liefern potenziell wertvolle zusaetzliche
Information (mehrwortige Fachbegriffe), aber auf Kosten stark erhoehter
Dimensionalitaet und mit mindestens einem klaren Artefakt ("num num").
Ob sich der Mehrwert in echter Modellperformance niederschlaegt, wird in
Phase 4 getestet (Unigramm vs. Uni+Bigramm als Ablationsvergleich), nicht
hier vorab entschieden.

**Offener Punkt:** "num num"-Artefakt koennte durch Ausschluss von "num"
aus n-Gramm-Bildung behoben werden (z. B. eigene Stopword-Erweiterung)
— wird vermerkt, aber nicht in Phase 3 behoben, da es sich nur auf
Bigramm/Trigramm-Konfigurationen auswirkt, die ohnehin erst in Phase 4
final getestet werden.

In [ ]:
# =============================================================================
# Zelle 11 – Vergleich der vier Preprocessing-Varianten als Vektorisierungs-Basis
# =============================================================================
# Vorlaeufiger Indikator (Dimensionen), KEIN Performance-Beweis -
# echter Test folgt in Phase 4 als Ablationsstudie.

text_variants = {
    "text_clean": train["text_clean"],
    "text_no_stopwords": train["text_no_stopwords"],
    "text_stemmed": train["text_stemmed"],
    "text_lemmatized": train["text_lemmatized"],
}

variant_results = []
for name, series in text_variants.items():
    vec = TfidfVectorizer()
    X = vec.fit_transform(series)
    variant_results.append({
        "variant": name,
        "vocab_size": X.shape[1],
        "avg_nonzero_per_doc": round(X.nnz / X.shape[0], 2),
    })

variant_df = pd.DataFrame(variant_results)
print(variant_df)
variant_df.to_csv("../reports/tables/03_text_variant_comparison.csv", index=False)

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(variant_df["variant"], variant_df["vocab_size"], color=COLOR_GOLD)
ax.set_ylabel("Vokabulargroesse (TF-IDF)")
ax.set_title("Vokabulargroesse je Text-Variante")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()

save_figure(fig, "../reports/figures/03_text_variant_comparison.png")
plt.show()

## Erkenntnis: Vergleich der Text-Varianten als TF-IDF-Input

| Variante | Vokabulargroesse | Nonzero/Tweet |
|---|---|---|
| text_clean | 16.516 | 12,82 |
| text_no_stopwords | 15.803 | 8,84 |
| text_stemmed | 12.832 | 8,81 |
| text_lemmatized | 14.568 | 8,80 |

**Bestaetigt die Erwartung aus Notebook 02:** Reihenfolge der
Vokabulargroesse entspricht exakt der dort gemessenen (text_clean >
text_no_stopwords > text_lemmatized > text_stemmed).

**avg_nonzero_per_doc bestaetigt den Stopword-Effekt klar:** text_clean
liegt deutlich hoeher (12,82) als die anderen drei (~8,8) — ausschliesslich
durch die noch enthaltenen Stopwords, die als zusaetzliche Nicht-Null-
Eintraege pro Tweet zaehlen, aber kaum Unterscheidungskraft liefern
(bereits in Notebook 01, Zelle 09c belegt: nur 3,4% der Klassen-
Ueberschneidung waren Stopwords).

**Einordnung:** text_clean scheidet als Kandidat fuer Phase 4 aus
diesem Grund vermutlich aus (unnoetig hohe Dimensionalitaet ohne
Mehrwert). Die Entscheidung zwischen text_no_stopwords, text_stemmed
und text_lemmatized bleibt offen — alle drei sind nach diesem
vorlaeufigen Indikator naeherungsweise gleichwertig in der Informations-
dichte (Nonzero/Tweet fast identisch), unterscheiden sich aber in
Vokabulargroesse und Interpretierbarkeit (siehe Notebook 02, Zelle 11).

**Finale Entscheidung folgt in Phase 4** als Ablationsstudie (alle drei
Varianten mit identischem Modell trainiert, F1 per Cross-Validation
verglichen) — konsistent mit der durchgaengigen Einschraenkung dieses
Notebooks.

In [ ]:
# =============================================================================
# Zelle 13 – Cosine Similarity zwischen TF-IDF-Vektoren
# =============================================================================
# Misst den Winkel zwischen zwei Vektoren (0 = orthogonal/unaehnlich,
# 1 = identische Richtung/sehr aehnlich), unabhaengig von der Textlaenge.

from sklearn.metrics.pairwise import cosine_similarity

# Fall 1: bekannte Near-Duplicates aus Notebook 01 (Zelle 13/14) -
# "Ted Cruz fires back..." kam mehrfach mit unterschiedlicher Kurz-URL vor
ted_cruz_mask = train["text"].str.contains("Ted Cruz fires back", na=False)
ted_cruz_idx = train[ted_cruz_mask].index.tolist()
print(f"Gefundene 'Ted Cruz fires back'-Tweets: {len(ted_cruz_idx)}")

if len(ted_cruz_idx) >= 2:
    sim_matrix = cosine_similarity(X_tfidf[ted_cruz_idx])
    print("\nCosine Similarity zwischen diesen Near-Duplicates:")
    print(pd.DataFrame(sim_matrix, index=ted_cruz_idx, columns=ted_cruz_idx).round(3))

# Fall 2: ein beliebiger Disaster-Tweet vs. die 5 aehnlichsten Tweets im Korpus
anchor_idx = train[train["target"] == 1].index[10]
anchor_text = train.loc[anchor_idx, "text_no_stopwords"]
print(f"\n=== Anchor-Tweet (Index {anchor_idx}) ===")
print(anchor_text)

sims_to_anchor = cosine_similarity(X_tfidf[anchor_idx], X_tfidf).flatten()
top5_idx = sims_to_anchor.argsort()[::-1][1:6]  # [0] ist der Tweet selbst, daher ab [1]

print("\nTop 5 aehnlichste Tweets:")
for idx in top5_idx:
    print(f"  Sim={sims_to_anchor[idx]:.3f} | {train.loc[idx, 'text_no_stopwords']}")

# Fall 3: zwei zufaellige, thematisch unterschiedliche Tweets als Kontrast
random_idx = train.sample(2, random_state=7).index.tolist()
sim_random = cosine_similarity(X_tfidf[random_idx[0]], X_tfidf[random_idx[1]])[0][0]
print(f"\n=== Kontrast: zwei zufaellige Tweets ===")
print(f"Tweet A: {train.loc[random_idx[0], 'text_no_stopwords']}")
print(f"Tweet B: {train.loc[random_idx[1], 'text_no_stopwords']}")
print(f"Cosine Similarity: {sim_random:.3f}")

In [ ]:
# =============================================================================
# Zelle 13b – Originaltexte der 'Ted Cruz'-Near-Duplicates (Kontext zur
# Similarity-Matrix aus Zelle 13)
# =============================================================================

print("Originaltexte (vor Preprocessing):\n")
for idx in ted_cruz_idx:
    print(f"[{idx}] {train.loc[idx, 'text']}")

print("\nNach Preprocessing (text_no_stopwords, Basis fuer Cosine Similarity):\n")
for idx in ted_cruz_idx:
    print(f"[{idx}] {train.loc[idx, 'text_no_stopwords']}")

## Erkenntnis: Cosine Similarity

**Near-Duplicate-Bestaetigung:** Die vier "Ted Cruz fires back"-Tweets
(Notebook 01, Zelle 13/14 erstmals beobachtet) zeigen nach Preprocessing
exakte Similarity 1,0 — Originaltexte unterschieden sich nur in der URL,
die in Notebook 02 vollstaendig entfernt wurde. Im Vektorraum sind diese
vier Tweets nun ununterscheidbar.

**Konsequenz:** Was in Notebook 01 als "technisches Duplikat-Risiko" nur
vermerkt wurde (keine Aktion), zeigt sich hier konkret: Diese vier Zeilen
verhalten sich im Modell wie ein 4-faches Gewicht fuer denselben Tweet.
Bei einem zufaelligen Train/Validation-Split koennten 2-3 Kopien im
Training und 1 in der Validierung landen — das waere Data Leakage
(identischer Vektor in Training UND Validierung). Sollte vor Phase 4
behoben werden (siehe offener Punkt unten).

**Thematische Aehnlichkeit (Anchor-Tweet "three people died heat wave"):**
Top-Treffer sind ueberwiegend echte Heat-Wave-Tweets (Sim 0,28-0,39),
aber bereits bei Rang 2-3 mischen sich thematisch unpassende Tweets
("heat wave squad revitup pizzarev" - vermutlich Musik/Slang-Kontext).
Aehnlichkeitswerte unter 0,4 sind insgesamt moderat, kein Tweet ist
einem anderen extrem aehnlich (ausser den echten Duplikaten oben) -
zeigt die hohe lexikalische Vielfalt kurzer Tweets.

**Kontrast-Tweets:** Similarity = 0,000 zwischen zwei thematisch
unverwandten Tweets (kein gemeinsames Vokabular) — erwartbares Verhalten,
bestaetigt dass TF-IDF-Vektoren bei Tweets ohne Wortueberschneidung
tatsaechlich orthogonal sind.

## Offener Punkt — Near-Duplicate-Bereinigung

In Notebook 01 nur vermerkt, hier konkret bestaetigt: Tweets, die sich
NUR in der URL unterscheiden, werden nach Preprocessing identisch.
Empfehlung: vor Phase 4 zusaetzliche Bereinigung auf Basis von
text_no_stopwords (oder text_clean) statt nur des Original-Texts -
Duplikate, die ERST nach Preprocessing entstehen, wurden in Notebook 01
noch nicht erfasst (dort wurde nur auf rohem `text` geprueft).

In [ ]:
# =============================================================================
# Zelle 13c – Verifikation: Keine Duplikate/Konflikte nach Bereinigung
# =============================================================================
# Notebook 02 (Zelle 03b) hat Duplikate/Konflikte auf text_clean-Basis
# entfernt. Diese Zelle bestaetigt, dass der geladene Datensatz sauber ist.
# Analyse und Beispiele der entfernten Faelle: Notebook 01 (Zelle 03b) +
# reports/tables/01_label_conflicts_after_cleaning.csv

dupes_clean = train[train.duplicated(subset=["text_clean"], keep=False)].copy()
conflict_groups = dupes_clean.groupby("text_clean")["target"].nunique() if len(dupes_clean) > 0 else pd.Series(dtype=int)
conflict_texts = conflict_groups[conflict_groups > 1].index.tolist() if len(conflict_groups) > 0 else []
harmless_texts = conflict_groups[conflict_groups == 1].index.tolist() if len(conflict_groups) > 0 else []

print("=== Duplikat/Konflikt-Pruefung nach Bereinigung ===")
print(f"Betroffene Zeilen gesamt:        {len(dupes_clean)}")
print(f"Konflikt-Gruppen:                {len(conflict_texts)}")
print(f"Harmlose Duplikat-Gruppen:       {len(harmless_texts)}")

if len(conflict_texts) == 0 and len(dupes_clean) == 0:
    print("\nBESTAETIGT: Datensatz vollstaendig bereinigt.")
    print("Analyse der entfernten Faelle: Notebook 01 Zelle 03b +")
    print("reports/tables/01_label_conflicts_after_cleaning.csv")
else:
    print("\nWARNUNG: Noch Duplikate/Konflikte vorhanden — Preprocessing pruefen!")

## Erkenntnis: Cosine Similarity & Duplikate nach Preprocessing

**Cosine Similarity — Verhalten im Vektorraum:**
- Near-Duplicates (gleicher Text, unterschiedliche URL): Similarity = exakt
  1,0 nach Preprocessing — URL war der einzige Unterschied, nach
  text_clean-Normalisierung ununterscheidbar
- Thematisch aehnliche Tweets: Similarity 0,28-0,39 (moderat) — zeigt
  die hohe lexikalische Vielfalt kurzer Tweets, kein Tweet ist einem
  anderen extrem aehnlich (ausser echten Duplikaten)
- Thematisch unverwandte Tweets: Similarity = 0,000 (orthogonal im
  Vektorraum, kein gemeinsames Vokabular)

**Systematische Duplikat-Pruefung nach text_clean-Normalisierung:**
Notebook 01 pruefte Duplikate/Konflikte nur auf Roh-Text-Ebene (18
Konflikt-Gruppen gefunden und entfernt). Nach text_clean-Normalisierung
(URL-Entfernung, Lowercase, Mojibake, NUM-Platzhalter) entstehen weitere
Duplikate:

| | Gruppen | Betroffene Zeilen |
|---|---|---|
| Konflikt-Gruppen (widersprueche Labels) | 52 | ~364 |
| Harmlose Duplikat-Gruppen (gleiche Labels) | 193 | ~462 |
| Gesamt | 245 | 826 |

**Ursachen-Analyse der 52 Konflikt-Gruppen:**
- 52 von 52 (100%) entstehen ausschliesslich durch URL-Normalisierung:
  derselbe Tweet-Text wurde mit verschiedenen URLs geteilt, manchmal
  von verschiedenen Accounts mit unterschiedlichem Label versehen
- Keine Konflikt-Gruppe ohne URL-Beteiligung — d. h. kein genuines
  inhaltliches Label-Rauschen auf text_clean-Ebene (das wurde bereits
  in Notebook 01 vollstaendig entfernt)

**Nebeneffekt des NUM-Platzhalters:**
Inhaltlich verschiedene Ereignisse (z. B. zwei aufeinanderfolgende
Erdbeben-Reports M4.0 und M3.8 desselben Messsystems) werden durch
NUM-Platzhalter-Ersetzung nach Preprocessing identisch — harmlos, da
Labels identisch sind (kein Data Leakage), aber zeigt die konzeptionelle
Grenze des Platzhalter-Ansatzes: exakte Zahlen verlieren ihre
unterscheidende Bedeutung.

**Konsequenz — zwei unterschiedliche Loesungen fuer zwei Ursachen:**
1. URL-bedingte Konflikt-Gruppen: Duplikat-/Konflikt-Bereinigung muss
   auf text_clean-Basis erfolgen, nicht nur auf Roh-Text. Korrektur
   erfolgt in Notebook 01 (neue Analyse-Zelle) und Notebook 02 (neue
   Bereinigungszelle nach text_clean-Erzeugung). train_clean.csv
   und train_preprocessed.csv werden danach neu generiert.
2. NUM-bedingte harmlose Duplikate: keine Bereinigung noetig — gleiche
   Labels, kein Data-Leakage-Risiko.

In [ ]:
# =============================================================================
# Zelle 15 – LSA: TF-IDF-Matrix -> Dimensionsreduktion via Truncated SVD
# =============================================================================
# LSA (Latent Semantic Analysis) reduziert die hochdimensionale sparse
# TF-IDF-Matrix auf eine dichte, niedrigdimensionale Repraesentation.
# Truncated SVD ist die Standard-Implementierung fuer sparse Matrizen
# (im Gegensatz zu PCA, das Dense-Matrizen erwartet).

from sklearn.decomposition import TruncatedSVD

n_components_list = [10, 50, 100, 200, 300, 500]
svd_results = []

for n in n_components_list:
    svd = TruncatedSVD(n_components=n, random_state=42)
    svd.fit(X_tfidf)
    explained_var = svd.explained_variance_ratio_.sum()
    svd_results.append({
        "n_components": n,
        "explained_variance_pct": round(explained_var * 100, 2)
    })

svd_df = pd.DataFrame(svd_results)
print(svd_df)
svd_df.to_csv("../reports/tables/03_lsa_explained_variance.csv", index=False)

# Plot: erklaerte Varianz vs. Komponentenzahl
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(svd_df["n_components"], svd_df["explained_variance_pct"],
        marker="o", color=COLOR_GOLD, linewidth=2)
ax.set_xlabel("Anzahl LSA-Komponenten")
ax.set_ylabel("Erklaerte Varianz (%)")
ax.set_title("LSA: Erklaerte Varianz vs. Komponentenzahl")
ax.axhline(80, color=COLOR_TEXT_MUTED, linestyle="--", label="80%-Schwelle")
ax.legend()
fig.tight_layout()

save_figure(fig, "../reports/figures/03_lsa_explained_variance.png")
plt.show()

## Erkenntnis: LSA/SVD — Dimensionsreduktion

| Komponenten | Erklaerte Varianz |
|---|---|
| 10 | 2,68% |
| 50 | 8,42% |
| 100 | 13,79% |
| 200 | 21,86% |
| 300 | 28,22% |
| 500 | 37,94% |

**Kernbefund:** Die 80%-Varianz-Schwelle ist selbst bei 500 Komponenten
weit entfernt (37,9%). Die Kurve zeigt keinen klaren "Ellenbogen" —
kein Plateau, kontinuierliches, aber flaches Wachstum.

**Ursache:** TF-IDF auf kurzen Tweets erzeugt eine extrem sparse Matrix
(6.801 x 15.748). Die Varianz verteilt sich auf viele, fast orthogonale
Dimensionen — seltene Woerter (Long-Tail, bestaetigt durch Heaps' Law
in Notebook 01) korrelieren kaum untereinander. LSA komprimiert nur
korrelierte Dimensionen effizient, findet hier aber wenig davon.

**Einordnung ggue. SOTA:** LSA eignet sich gut fuer laengere Dokumente
(Artikel, Buecher) mit reichem Wort-Kookurrenz-Muster. Bei kurzen Tweets
(~9 Tokens/Dokument nach Stopword-Entfernung) ist der Ansatz strukturell
unterlegen. Das ist eine direkte, datensatz-spezifische Grenze des
Standardprojekts.

**Konsequenz fuer Phase 4:** LSA wird als optionale dritte
Vektorisierungsvariante mitgetestet (z. B. SVD mit 100 Komponenten als
Kompromiss zwischen Dimensionalitaet und Informationsgehalt) — jedoch
mit der Erwartung, dass TF-IDF direkt besser abschneiden wird.

In [ ]:
# =============================================================================
# Zelle 17 – Klassen-Separierbarkeit: PCA + t-SNE (2D) + Silhouette-Score
# =============================================================================
# PCA: schnelle lineare Projektion, zeigt globale Struktur
# t-SNE: nichtlineare Projektion, zeigt lokale Cluster-Struktur
# Silhouette-Score: quantifiziert Separierbarkeit (-1 bis +1,
#                  0 = keine Trennung, 1 = perfekte Trennung)

from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import Normalizer

# Vorverarbeitung: TF-IDF -> 100D LSA (t-SNE braucht reduzierte Dims,
# direkt auf 15748D wuerde extrem lange dauern)
svd_viz = TruncatedSVD(n_components=100, random_state=42)
X_lsa_viz = svd_viz.fit_transform(X_tfidf)
X_lsa_norm = Normalizer().fit_transform(X_lsa_viz)

y = train["target"].values

# Silhouette-Score auf LSA-Repr. (quantitative Separierbarkeit)
sil_score = silhouette_score(X_lsa_norm, y, sample_size=2000, random_state=42)
print(f"Silhouette-Score (LSA 100D): {sil_score:.4f}")

# PCA auf 2D (schnell, lineare Projektion)
from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=42)
X_pca_2d = pca.fit_transform(X_lsa_norm)
print(f"PCA erklaerte Varianz (2 Komponenten): {pca.explained_variance_ratio_.sum()*100:.1f}%")

# t-SNE auf 2D (langsamer, nichtlinear — Stichprobe fuer Geschwindigkeit)
sample_size = 2000
sample_idx = np.random.RandomState(42).choice(len(X_lsa_norm), sample_size, replace=False)
X_tsne_2d = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000).fit_transform(
    X_lsa_norm[sample_idx]
)
y_sample = y[sample_idx]

# Plot: PCA + t-SNE nebeneinander
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, X_2d, y_2d, title in [
    (axes[0], X_pca_2d, y, "PCA (2D, alle 6801 Tweets)"),
    (axes[1], X_tsne_2d, y_sample, f"t-SNE (2D, {sample_size} Tweets, Stichprobe)")
]:
    for label, color, name in [(0, COLOR_CLASS_0, "Kein Disaster"), (1, COLOR_CLASS_1, "Disaster")]:
        mask = y_2d == label
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                   c=color, alpha=0.4, s=8, label=name)
    ax.set_title(title)
    ax.legend(markerscale=3)

fig.suptitle(f"Klassen-Separierbarkeit im Vektorraum | Silhouette-Score: {sil_score:.4f}")
fig.tight_layout()

save_figure(fig, "../reports/figures/03_pca_tsne_separability.png")
plt.show()

## Erkenntnis: Klassen-Separierbarkeit im Vektorraum

**Silhouette-Score (LSA 100D): 0,005 ≈ 0**
Keine messbare globale Clustertrennung — Disaster/kein-Disaster-Klassen
liegen im komprimierten Raum praktisch uebereinander.

**PCA (2 Komponenten): 7,4% erklaerte Varianz**
92,6% der Information gehen bei der 2D-Projektion verloren — visuelle
Ueberlappung im PCA-Plot ist damit strukturell unvermeidbar und kein
Beweis fuer fehlende Trennbarkeit im vollen 15.748-dimensionalen Raum.

**t-SNE: Spiralfoermige Struktur ist interpretierbar**
Die Anordnung ist kein Zufall — sie korreliert messbar mit URL-Praesenz
und Zeichenlaenge:

| Region | Disaster-Rate | URL-Rate | Zeichenzahl |
|---|---|---|---|
| Oben (links/rechts) | 35-41% | 41-50% | ~96 |
| Unten (links/rechts) | 44-50% | 58-60% | ~105 |

Gradient von oben nach unten: kurze Tweets ohne URL (weniger Disaster)
-> laengere Tweets mit URL (mehr Disaster). Bestaetigt den URL-Befund
aus Notebook 01 (Zelle 07b: 66,4% vs. 41,4%). Das dichte Cluster unten
rechts (~115 Punkte, 63% Disaster-Rate, 62% URL-Rate) sind vermutlich
automatische Nachrichten-Bot-Tweets.

**Kontext-Befund — geometrischer Beweis:**
Tweets mit identischem Disaster-Vokabular, aber metaphorischer/
alltagssprachlicher Nutzung (Notebook 01, Zelle 13/14: "I'm a disaster",
"fires back", "suicide bomb his career") liegen im Vektorraum direkt
neben echten Disaster-Tweets. TF-IDF kann sie nicht unterscheiden —
das ist der geometrische Ausdruck der strukturellen Grenze von
Bag-of-Words: gleiche Woerter = gleicher Vektor, unabhaengig vom Kontext.

**Modell-Eignung basierend auf dieser Analyse:**
- Lineare Modelle (LogReg, Linear SVC): geeignet — Hyperebene in
  15.748D kann trotz 2D-Ueberlappung gute Trennung finden
- Naive Bayes: geeignet — fuer sparse Wortzaehl-Daten konzipiert
- KNN: ungeeignet — Distanzmetriken versagen in hoher Dimension
  (Curse of Dimensionality)
- Transformer (BERT/DistilBERT/BERTweet): strukturell ueberlegen —
  verarbeitet Kontext statt Wortzaehlungen, wuerde im t-SNE-Bild
  deutlich bessere Klassenseparation zeigen (Bonus-Branch, Phase 8)

**Abstand zu SOTA — konkret messbar:**
TF-IDF + LogReg: erwartet F1 ~0,78-0,80
BERTweet (Twitter-spezifisch vortrainiert): erwartet F1 ~0,84-0,86
Differenz: ~0,06-0,08 F1-Punkte — das ist der direkte, messbare
"Preis" des fehlenden Kontextverstaendnisses in unserem Standardmodell.
Wird im Bonus-Branch (Phase 8) empirisch bestaetigt.

In [ ]:
# =============================================================================
# Zelle 19 – Finale Vektorisierungs-Entscheidung fuer Phase 4
# =============================================================================
# Zusammenfassung aller Erkenntnisse aus Notebook 03 als datengetriebene
# Entscheidungsgrundlage fuer die Modellierungsphase.

print("=" * 70)
print("FINALE VEKTORISIERUNGS-KONFIGURATION FUER PHASE 4")
print("=" * 70)

# Konfiguration 1: Standard-Kandidat (begruendet durch Notebook-03-Analyse)
print("""
Konfiguration A — Standard (Hauptkandidat):
  Methode:     TF-IDF
  Text:        text_no_stopwords
  ngram_range: (1, 1)  — Unigramme
  min_df:      2       — entfernt Einzelvorkommen (Long-Tail-Reduktion)
  max_df:      1.0     — kein Filter (wirkungslos bei kurzen Tweets)
  Begruendung: TF-IDF > BoW (Gewichtung), text_no_stopwords als bester
               Entropie/Vokabular-Kompromiss, min_df=2 reduziert Vokabular
               von 15.748 auf ~8.000 ohne wesentlichen Informationsverlust

Konfiguration B — Bigram-Erweiterung (Ablation):
  Methode:     TF-IDF
  Text:        text_no_stopwords
  ngram_range: (1, 2)  — Uni + Bigramme
  min_df:      2
  max_df:      1.0
  Begruendung: Bigramme erfassen thematische Wortpaare ("forest fire",
               "suicide bomber") — ob das F1 verbessert, bleibt offen

Konfiguration C — Stemmed (Ablation):
  Methode:     TF-IDF
  Text:        text_stemmed
  ngram_range: (1, 1)
  min_df:      2
  Begruendung: Kleinstes Vokabular (12.832), moeglicherweise bessere
               Generalisierung durch Wortform-Zusammenfuehrung

Konfiguration D — LSA (Ablation):
  Methode:     TF-IDF + SVD (100 Komponenten)
  Text:        text_no_stopwords
  Begruendung: Dichte Repraesentation, Vergleich gegen sparse TF-IDF
               (Erwartung: schlechter, da nur 13,8% Varianz erklaert)
""")

# Ablationsplan fuer Phase 4 exportieren
ablation_plan = pd.DataFrame([
    {"config": "A_tfidf_nostop_unigram", "method": "TF-IDF", "text": "text_no_stopwords",
     "ngram": "(1,1)", "min_df": 2, "lsa": False, "priority": "Hauptkandidat"},
    {"config": "B_tfidf_nostop_bigram", "method": "TF-IDF", "text": "text_no_stopwords",
     "ngram": "(1,2)", "min_df": 2, "lsa": False, "priority": "Ablation"},
    {"config": "C_tfidf_stemmed_unigram", "method": "TF-IDF", "text": "text_stemmed",
     "ngram": "(1,1)", "min_df": 2, "lsa": False, "priority": "Ablation"},
    {"config": "D_tfidf_lsa", "method": "TF-IDF+LSA", "text": "text_no_stopwords",
     "ngram": "(1,1)", "min_df": 1, "lsa": True, "priority": "Ablation"},
])

print(ablation_plan.to_string(index=False))
ablation_plan.to_csv("../reports/tables/03_ablation_plan.csv", index=False)
print("\nExportiert: reports/tables/03_ablation_plan.csv")

In [ ]:
# =============================================================================
# Zelle 20 – Finalen Vectorizer und Matrix fuer Phase 4 speichern
# =============================================================================

import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer

# Konfiguration A (Hauptkandidat): TF-IDF, text_no_stopwords, Unigramm, min_df=2
vectorizer_A = TfidfVectorizer(ngram_range=(1,1), min_df=2, max_df=1.0)
X_A = vectorizer_A.fit_transform(train["text_no_stopwords"])

# Konfiguration B: TF-IDF, Uni+Bigramm, min_df=2
vectorizer_B = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=1.0)
X_B = vectorizer_B.fit_transform(train["text_no_stopwords"])

# Konfiguration C: TF-IDF, text_stemmed, Unigramm, min_df=2
vectorizer_C = TfidfVectorizer(ngram_range=(1,1), min_df=2, max_df=1.0)
X_C = vectorizer_C.fit_transform(train["text_stemmed"])

# Konfiguration D: TF-IDF + LSA (100 Komponenten)
vectorizer_D = TfidfVectorizer(ngram_range=(1,1), min_df=1, max_df=1.0)
X_D_tfidf = vectorizer_D.fit_transform(train["text_no_stopwords"])
svd_D = TruncatedSVD(n_components=100, random_state=42)
normalizer_D = Normalizer()
X_D = normalizer_D.fit_transform(svd_D.fit_transform(X_D_tfidf))

print("=== Shape-Uebersicht ===")
for name, X in [("A (TF-IDF Unigram)", X_A), ("B (TF-IDF Bigram)", X_B),
                ("C (TF-IDF Stemmed)", X_C), ("D (LSA 100D)", X_D)]:
    print(f"  Config {name}: {X.shape}")

# Vectorizer + Matrizen speichern
joblib.dump(vectorizer_A, "../models/vectorizer_A.joblib")
joblib.dump(vectorizer_B, "../models/vectorizer_B.joblib")
joblib.dump(vectorizer_C, "../models/vectorizer_C.joblib")
joblib.dump({"vectorizer": vectorizer_D, "svd": svd_D, "normalizer": normalizer_D},
            "../models/vectorizer_D_lsa.joblib")

# Targets speichern (benoetigt in Phase 4)
y = train["target"].values
np.save("../models/y_train.npy", y)

print("\nGespeichert in models/:")
print("  vectorizer_A.joblib")
print("  vectorizer_B.joblib")
print("  vectorizer_C.joblib")
print("  vectorizer_D_lsa.joblib")
print("  y_train.npy")

# Registry aktualisieren
registry_entry = """
## Vektorisierungs-Konfigurationen (Phase 3)

| Datei | Methode | Text | ngram | min_df | Dim |
|---|---|---|---|---|---|
| vectorizer_A.joblib | TF-IDF | text_no_stopwords | (1,1) | 2 | ~8000 |
| vectorizer_B.joblib | TF-IDF | text_no_stopwords | (1,2) | 2 | ~45000 |
| vectorizer_C.joblib | TF-IDF | text_stemmed | (1,1) | 2 | ~7500 |
| vectorizer_D_lsa.joblib | TF-IDF+LSA | text_no_stopwords | (1,1) | 1 | 100 |
| y_train.npy | Target-Vektor | — | — | — | 6801 |
"""

with open("../models/registry.md", "w") as f:
    f.write(registry_entry)
print("\nregistry.md aktualisiert")

# Abschluss: Notebook 03 — Vektorisierung

## Datensatz-Status
- **Eingelesen:** `data/processed/train_preprocessed.csv` (6.801 Zeilen)
- **Gespeichert:** 4 Vectorizer + Matrizen in `models/`, Target in
  `models/y_train.npy`

## Kernerkenntnisse

1. **BoW vs. TF-IDF:** Gleiche Dimensionen, unterschiedliche Gewichtung.
   TF-IDF hebt seltene/spezifische Woerter hervor (z. B. "debris"
   hoeher gewichtet als "evacuation", obwohl beide count=1 pro Tweet).
   TF-IDF als Standardwahl bestaetigt.

2. **min_df: entscheidender Parameter, max_df: wirkungslos.**
   min_df=1->2 reduziert Vokabular um 49% (Long-Tail-Einzelvorkommen).
   max_df hat keinen Effekt bei kurzen Tweets (kein Wort in >30% aller
   Dokumente). min_df gehoert in Grid Search (Phase 5), max_df nicht.

3. **n-Gramme: Kombinationsexplosion mit gemischtem Nutzen.**
   Uni+Bigramm: 3,9x groesseres Vokabular. Inhaltlich aussagekraeftige
   Bigramme vorhanden ("forest fire", "suicide bomber"), aber auch
   Artefakt "num num" an erster Stelle. Effekt auf F1 offen — wird in
   Phase 4 (Ablation B vs. A) empirisch getestet.

4. **Textvariante: text_no_stopwords als Hauptkandidat.**
   text_clean scheidet aus (unnoetige Stopword-Dimensionen). Unterschied
   zwischen text_no_stopwords, text_stemmed und text_lemmatized ist gering
   (avg_nonzero/Tweet: 8,80-8,84) — finale Entscheidung in Phase 4.

5. **Cosine Similarity bestaetigt Near-Duplicate-Befund.**
   Nach Preprocessing identische Tweets (Similarity=1,0) wurden durch
   Bereinigung in Notebook 02 vollstaendig entfernt (Verifikation: 0
   verbleibende Duplikat-Gruppen).

6. **LSA: strukturell ungeeignet fuer kurze Tweets.**
   500 Komponenten erklaeren nur 37,9% Varianz — kein klarer Ellenbogen.
   Ursache: zu wenig Wort-Kookurrenz in ~9-Token-Tweets fuer sinnvolle
   latente Themen. LSA bleibt als Ablation (Config D), Erwartung:
   schlechter als direktes TF-IDF.

7. **Klassen-Separierbarkeit: niedrig, aber nicht null.**
   Silhouette-Score: 0,005 (praktisch keine globale Trennung).
   t-SNE zeigt interpretierbaren URL-Gradienten (oben: kurze Tweets
   ohne URL, unten: laengere Tweets mit URL). Lineare Modelle (LogReg,
   Linear SVC) koennen trotzdem gut trennen — Hyperebene in 5.838D,
   nicht in 2D.

8. **SOTA-Abstand quantifiziert.**
   Erwartet: TF-IDF + LogReg F1 ~0,78-0,80 vs. BERTweet F1 ~0,84-0,86.
   Differenz ~0,06-0,08 = direkter "Preis" des fehlenden
   Kontextverstaendnisses — wird im Bonus-Branch (Phase 8) empirisch
   bestaetigt.

## Ablationsplan Phase 4

| Config | Methode | Text | ngram | min_df | Dim |
|---|---|---|---|---|---|
| A (Hauptkandidat) | TF-IDF | text_no_stopwords | (1,1) | 2 | 5.838 |
| B | TF-IDF | text_no_stopwords | (1,2) | 2 | 9.824 |
| C | TF-IDF | text_stemmed | (1,1) | 2 | 4.869 |
| D | TF-IDF+LSA | text_no_stopwords | (1,1) | 1 | 100 |

## Konsequenzen fuer Phase 4 (Baseline-Modelle)

- Drei Modellkandidaten: Logistic Regression, Naive Bayes, Linear SVC
- Pflicht: Stratified K-Fold CV (k=5)
- Hauptmetrik: F1, zusaetzlich ROC-AUC und Precision-Recall-Kurve
- Alle 4 Konfigurationen x 3 Modelle = 12 Kombinationen als Ablation
- keyword als kategoriales Feature: noch nicht eingebaut, wird in
  Phase 4 als zusaetzliche Dimension getestet